# Phase 3 — Master Analytical Panel & Exploratory Data Analysis

## Eastern Cape Municipal Electoral Dynamics

This notebook integrates the completed election and demographic phases into the project's **master analytical panel** and performs the main exploratory analysis.

### Phase 3 objectives

1. Merge Phase 1 election outcomes, competition measures and selected party shares.
2. Merge Phase 2 temporally aligned demographic features.
3. Validate the municipality × election-year panel.
4. Analyse voter turnout across election years and municipalities.
5. Examine descriptive associations between turnout, political competition and demographics.
6. Produce reusable tables and figures for modelling, the dashboard and presentation.
7. Create a **leakage-safe modelling view** for Phase 4.

### Interpretation rule

All relationships in this notebook are **descriptive associations**. Municipality-level correlations do not establish individual voter behaviour or causality.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Run this notebook from inside the DIRISA project.")

PROJECT_ROOT = find_project_root()

PHASE1_DIR = PROJECT_ROOT / "data" / "processed" / "elections" / "phase1"
PHASE2_DIR = PROJECT_ROOT / "data" / "processed" / "demographics" / "phase2"
PANEL_DIR = PROJECT_ROOT / "data" / "processed" / "panels" / "phase3"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures" / "phase3"

PANEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ELECTION_FILE = PHASE1_DIR / "municipality_election_panel_2000_2021.csv"
PARTY_FILE = PHASE1_DIR / "party_shares_selected_wide_2000_2021.csv"

# Support both the original Phase 2 filename and the consolidated Phase 2 filename.
DEMOGRAPHIC_CANDIDATES = [
    PHASE2_DIR / "demographics_historical_aligned_2000_2021.csv",
    PHASE2_DIR / "demographics_aligned_to_elections_2000_2021.csv",
]
DEMOGRAPHIC_FILE = next((p for p in DEMOGRAPHIC_CANDIDATES if p.exists()), None)

print("Project root:", PROJECT_ROOT)
print("Phase 1 election file:", ELECTION_FILE.exists())
print("Phase 1 party file:", PARTY_FILE.exists())
print("Phase 2 demographic file:", DEMOGRAPHIC_FILE)

if not ELECTION_FILE.exists():
    raise FileNotFoundError(f"Missing Phase 1 output: {ELECTION_FILE}")
if not PARTY_FILE.exists():
    raise FileNotFoundError(f"Missing Phase 1 output: {PARTY_FILE}")
if DEMOGRAPHIC_FILE is None:
    raise FileNotFoundError(
        "Missing Phase 2 historical demographic output. Run Phase 2 first."
    )


## 1. Load authoritative Phase 1 and Phase 2 outputs

Unlike the earlier exploratory teammate notebook, Phase 3 does **not** concatenate the old `clean_res11`, `clean_res16` and `clean_res21` files. It starts from the validated Phase 1 municipality-election panel and Phase 2 demographic output.


In [ ]:
elections = pd.read_csv(ELECTION_FILE)
party = pd.read_csv(PARTY_FILE)
demographics = pd.read_csv(DEMOGRAPHIC_FILE)

print("Election panel:", elections.shape)
print("Selected party shares:", party.shape)
print("Demographics:", demographics.shape)

display(elections.head())
display(demographics.head())


## 2. Build the master analytical panel

In [ ]:
# Validate join keys before merging.
for name, frame in {
    "elections": elections,
    "party": party,
    "demographics": demographics,
}.items():
    assert {"MuniCode", "Year"}.issubset(frame.columns)
    assert not frame.duplicated(["MuniCode", "Year"]).any(), (
        f"Duplicate MuniCode-Year keys in {name}"
    )

master = elections.merge(
    party,
    on=["MuniCode", "Year"],
    how="left",
    validate="one_to_one",
)

master = master.merge(
    demographics,
    on=["MuniCode", "Year"],
    how="left",
    validate="one_to_one",
)

master = master.sort_values(["Year", "MuniCode"]).reset_index(drop=True)

print("Master panel shape:", master.shape)
display(master.head())


## 3. Master-panel quality control

Expected election rows:

- 2000: 32 municipalities
- 2006: 33
- 2011: 33
- 2016: 33
- 2021: 33
- Total: **164 municipality-election observations**

The 2000 difference is inherited from Phase 1's harmonised analytical geography. Historical demographic variables should be available for 2011 onward and intentionally unavailable before 2011.


In [ ]:
year_counts = master.groupby("Year").size()
expected_counts = {2000: 32, 2006: 33, 2011: 33, 2016: 33, 2021: 33}

qc_rows = [
    ["Master rows", len(master), 164],
    ["Duplicate MuniCode-Year keys", master.duplicated(["MuniCode", "Year"]).sum(), 0],
    ["Missing party-share joins", master[["ANC", "DA", "EFF", "UDM", "ATM"]].isna().all(axis=1).sum(), 0],
    ["Invalid turnout rows", ((master["Turnout_%"] < 0) | (master["Turnout_%"] > 100)).sum(), 0],
    ["Invalid ENP rows", (master["ENP"] < 1).sum(), 0],
    ["Negative margin rows", (master["Margin_pts"] < 0).sum(), 0],
]

for year, expected in expected_counts.items():
    qc_rows.append([
        f"{year} rows",
        int(year_counts.get(year, 0)),
        expected
    ])

post2011 = master[master["Year"] >= 2011]
pre2011 = master[master["Year"] < 2011]
demo_cols = ["Pop", "U15", "O65", "Med", "NoSch", "Matric", "Higher", "Formal", "Water", "Elec"]

qc_rows += [
    ["2011+ rows with complete demographics", post2011[demo_cols].notna().all(axis=1).sum(), 99],
    ["Pre-2011 rows with demographics intentionally unavailable", pre2011[demo_cols].isna().all(axis=1).sum(), 65],
]

qc = pd.DataFrame(qc_rows, columns=["Check", "Actual", "Expected"])
qc["Pass"] = qc["Actual"] == qc["Expected"]

display(qc)
assert qc["Pass"].all(), "Phase 3 QC gate failed."
print("PASS: master analytical panel validated.")


## 4. Turnout descriptive statistics

Two summaries are useful:

- **Municipality mean turnout**: every municipality receives equal weight.
- **Registered-voter-weighted turnout**: total votes cast divided by total registered voters for the municipalities in that election year.

The second measure better represents aggregate participation across the panel, while the first describes the typical municipality.


In [ ]:
turnout_summary = (
    master.groupby("Year")["Turnout_%"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)

weighted = (
    master.groupby("Year")[["VotesCast", "RegisteredVoters"]]
    .sum()
    .reset_index()
)
weighted["WeightedTurnout_%"] = (
    100 * weighted["VotesCast"] / weighted["RegisteredVoters"]
)

turnout_summary = turnout_summary.merge(
    weighted[["Year", "WeightedTurnout_%"]],
    on="Year",
    how="left",
)

turnout_summary = turnout_summary.rename(columns={
    "count": "Municipalities",
    "mean": "MeanTurnout_%",
    "median": "MedianTurnout_%",
    "std": "SDTurnout_%",
    "min": "MinTurnout_%",
    "max": "MaxTurnout_%",
})

display(turnout_summary.round(2))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(turnout_summary["Year"], turnout_summary["MeanTurnout_%"], marker="o", label="Municipality mean")
ax.plot(turnout_summary["Year"], turnout_summary["WeightedTurnout_%"], marker="o", label="Registered-voter weighted")
ax.set_title("Eastern Cape Municipal Turnout by Local Election Year")
ax.set_xlabel("Election year")
ax.set_ylabel("Turnout (%)")
ax.set_xticks(turnout_summary["Year"])
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "turnout_by_year.png", dpi=180, bbox_inches="tight")
plt.show()


## 5. Municipality-level turnout change

In [ ]:
turnout_pivot = master.pivot(
    index="MuniCode",
    columns="Year",
    values="Turnout_%"
)

# Focus the teammate's original comparison on the fully harmonised 2011–2021 period.
turnout_pivot["Change_2011_2021_pts"] = turnout_pivot[2021] - turnout_pivot[2011]
turnout_change = (
    turnout_pivot[[2011, 2016, 2021, "Change_2011_2021_pts"]]
    .dropna()
    .sort_values("Change_2011_2021_pts")
    .reset_index()
)

display(turnout_change)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
plot_change = turnout_change.sort_values("Change_2011_2021_pts")
ax.barh(plot_change["MuniCode"], plot_change["Change_2011_2021_pts"])
ax.axvline(0, linewidth=1)
ax.set_title("Municipality Turnout Change: 2011 to 2021")
ax.set_xlabel("Change in turnout (percentage points)")
ax.set_ylabel("Municipality code")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "turnout_change_2011_2021.png", dpi=180, bbox_inches="tight")
plt.show()


## 6. Political competition and turnout

Phase 1 already calculated:

- **ENP** — Effective Number of Parties, from the full party vote distribution.
- **Margin_pts** — percentage-point gap between the first- and second-placed party.
- **NumberOfParties** — number of parties represented in the municipal PR vote data.

These same-election variables are appropriate here for **descriptive association analysis**.

They must **not** later be used as pre-election predictors of turnout for that same election because they are only observed after votes are counted. Phase 4 will use lagged competition variables instead.


In [ ]:
competition_vars = ["Turnout_%", "ENP", "Margin_pts", "NumberOfParties"]
competition_corr = master[competition_vars].corr()

display(competition_corr.round(3))


In [ ]:
for x, title, filename in [
    ("ENP", "Turnout and Effective Number of Parties", "turnout_vs_enp.png"),
    ("Margin_pts", "Turnout and Victory Margin", "turnout_vs_margin.png"),
    ("NumberOfParties", "Turnout and Number of Parties", "turnout_vs_number_of_parties.png"),
]:
    fig, ax = plt.subplots(figsize=(7, 5))
    for year, group in master.groupby("Year"):
        ax.scatter(group[x], group["Turnout_%"], label=str(year), alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel(x)
    ax.set_ylabel("Turnout (%)")
    ax.legend(title="Year")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()


## 7. Demographic associations with turnout

Demographic analysis is restricted to **2011–2021**, where Phase 2 provides municipality features.

Remember:

- 2011 values are observed Census values.
- 2016 and 2021 values are interpolated between the 2011 and 2022 municipality snapshots.
- These are municipality-level characteristics, not individual voter attributes.


In [ ]:
analysis_2011_2021 = master[master["Year"].isin([2011, 2016, 2021])].copy()

demographic_vars = [
    "Pop", "U15", "O65", "Med", "NoSch",
    "Matric", "Higher", "Formal", "Water", "Elec"
]

demo_corr = (
    analysis_2011_2021[["Turnout_%"] + demographic_vars]
    .corr()["Turnout_%"]
    .drop("Turnout_%")
    .sort_values()
    .rename("CorrelationWithTurnout")
    .reset_index()
    .rename(columns={"index": "Variable"})
)

display(demo_corr.round(3))


In [ ]:
for x, title, filename in [
    ("U15", "Under-15 Population Share and Turnout", "turnout_vs_u15.png"),
    ("O65", "Older Population Share and Turnout", "turnout_vs_o65.png"),
    ("Matric", "Matric Share and Turnout", "turnout_vs_matric.png"),
    ("Higher", "Higher-Education Share and Turnout", "turnout_vs_higher_education.png"),
    ("Water", "Water Access and Turnout", "turnout_vs_water.png"),
    ("Elec", "Electricity Access and Turnout", "turnout_vs_electricity.png"),
]:
    fig, ax = plt.subplots(figsize=(7, 5))
    for year, group in analysis_2011_2021.groupby("Year"):
        ax.scatter(group[x], group["Turnout_%"], label=str(year), alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel(x)
    ax.set_ylabel("Turnout (%)")
    ax.legend(title="Year")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()


## 8. Combined association table

In [ ]:
association_vars = demographic_vars + ["ENP", "Margin_pts", "NumberOfParties"]

association_table = (
    analysis_2011_2021[["Turnout_%"] + association_vars]
    .corr()["Turnout_%"]
    .drop("Turnout_%")
    .sort_values()
    .rename("PearsonCorrelationWithTurnout")
    .reset_index()
    .rename(columns={"index": "Variable"})
)

display(association_table.round(3))


In [ ]:
corr_vars = ["Turnout_%", "ENP", "Margin_pts", "U15", "O65", "Matric", "Higher", "Formal", "Water", "Elec"]
corr_matrix = analysis_2011_2021[corr_vars].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix.values, aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_vars)))
ax.set_yticks(range(len(corr_vars)))
ax.set_xticklabels(corr_vars, rotation=45, ha="right")
ax.set_yticklabels(corr_vars)

for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

ax.set_title("Correlation Matrix: Turnout, Competition and Demographics (2011–2021)")
fig.colorbar(im, ax=ax, label="Pearson correlation")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "turnout_correlation_matrix.png", dpi=180, bbox_inches="tight")
plt.show()


## 9. Leakage-safe Phase 4 modelling view

Phase 4 will predict turnout using information that could plausibly be known before the target election.

Therefore this view retains:

- previous turnout;
- previous ENP;
- previous victory margin;
- previous registered voters;
- contemporaneous demographic estimates aligned by Phase 2.

It deliberately excludes same-election `ENP`, `Margin_pts`, `TopShare_%`, party shares and other election outcomes from the predictor set.

For the historical 2021 backtest, 2011 and 2016 can form training observations and 2021 can be held out for testing.


In [ ]:
model_feature_cols = [
    "MuniCode", "Year", "Turnout_%",
    "PreviousRegisteredVoters", "PreviousTurnout_%",
    "PreviousENP", "PreviousMargin_pts", "PreviousTopShare_%",
    "Pop", "U15", "O65", "Med", "NoSch",
    "Matric", "Higher", "Formal", "Water", "Elec",
    "DemographicStatus"
]

model_ready = master.loc[
    master["Year"].isin([2011, 2016, 2021]),
    model_feature_cols
].copy()

assert len(model_ready) == 99
assert model_ready[["PreviousTurnout_%", "PreviousENP", "PreviousMargin_pts"]].notna().all().all()
assert model_ready[demo_cols].notna().all().all()

print("Model-ready rows:", len(model_ready))
print("Training years:", [2011, 2016])
print("Historical holdout year:", 2021)
display(model_ready.head())


## 10. Save Phase 3 outputs

The master panel is the authoritative integrated dataset for downstream work. The old prototype `data/processed/panels/panel.csv` should no longer be treated as the modelling source.


In [ ]:
outputs = {
    "master_panel_2000_2021.csv": master,
    "analysis_panel_2011_2021.csv": analysis_2011_2021,
    "turnout_model_ready_2011_2021.csv": model_ready,
    "turnout_summary_by_year.csv": turnout_summary,
    "municipality_turnout_change_2011_2021.csv": turnout_change,
    "turnout_association_table_2011_2021.csv": association_table,
    "phase3_quality_control.csv": qc,
}

for filename, frame in outputs.items():
    path = PANEL_DIR / filename
    frame.to_csv(path, index=False)
    print("Saved:", path.relative_to(PROJECT_ROOT), "|", len(frame), "rows")

print("\nFigures saved to:", FIGURE_DIR.relative_to(PROJECT_ROOT))
print("Phase 3 complete.")


## Phase 3 completion note

Phase 3 has now:

- integrated Phase 1 and Phase 2;
- preserved the complete 2000–2021 electoral history;
- created the 2011–2021 demographic analysis subset;
- produced turnout, competition and demographic EDA;
- retained the teammate's useful turnout-analysis objectives;
- prevented same-election competition measures from leaking into the future turnout model;
- prepared the 99-row historical modelling dataset for Phase 4.

### Next phase

**Phase 4 — Turnout Model**

The central historical evaluation will hold out **2021** and compare the model against simple baselines before any 2026 scenario is produced.
